<a href="https://www.kaggle.com/code/mohamedaminebayar/102-mole-cancer-classifier?scriptVersionId=311844670" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🔬 Mole Cancer Risk Classifier
**Fast.ai Image Classifier — Is this mole dangerous?**

This notebook builds a binary image classifier using fast.ai's `vision_learner` to predict whether a skin mole/lesion is **benign** or **malignant**.

Dataset: [Skin Cancer: Malignant vs Benign](https://www.kaggle.com/datasets/fanconic/skin-cancer-malignant-vs-benign)

> ⚠️ **Medical disclaimer:** This model is for learning purposes only and is not a medical diagnostic tool. Always consult a dermatologist for real assessments.

## Step 1 — Setup

In [ ]:
# fast.ai is pre-installed on Kaggle — no pip install needed!
from fastai.vision.all import *

## Step 2 — Load the Dataset

**Skin Cancer: Malignant vs Benign** dataset from Kaggle Datasets.
The folder structure is already split into `train/` and `test/`, with subfolders:
- `benign/` — harmless moles
- `malignant/` — potentially cancerous lesions

In [ ]:
from pathlib import Path
import os

for root, dirs, files in os.walk('/kaggle/input'):
    if 'benign' in dirs or 'malignant' in dirs:
        print(root)
        break

In [ ]:
path = Path('/kaggle/input/datasets/fanconic/skin-cancer-malignant-vs-benign')

# Check what's in the dataset
print('Top-level folders:', [f.name for f in path.ls()])
print('\nTraining classes:', [f.name for f in (path/'train').ls()])

# Count images per class
for cls in (path/'train').ls():
    n = len(list(cls.glob('*.jpg')) + list(cls.glob('*.png')))
    print(f'  {cls.name}: {n} images')

## Step 3 — Create DataLoaders

fast.ai's `ImageDataLoaders.from_folder` handles everything:
- Reading images from subfolders (subfolder name = label)
- Splitting into train / validation sets
- Resizing and augmenting images on the fly

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=partial(get_files, extensions=['.jpg','.jpeg','.png']),
    splitter=RandomSplitter(0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(224)
).dataloaders(path/'train')

#### Step 4 — Visualise the Data

It's a must to look at the data (or at least a sample) before training!

In [ ]:
dls.show_batch(max_n=9, figsize=(12, 12))

## Step 5 — Build & Train the Model

**ResNet-34** is pretrained on ImageNet and fine-tuned for this specific task.

Metrics:
- `error_rate` — how often we get it wrong
- `RocAuc` — especially useful for imbalanced medical datasets
- `F1Score` — balances precision and recall (important: false negatives are dangerous!)

In [ ]:
learn = vision_learner(
    dls,
    resnet34,
    metrics=[error_rate, RocAucBinary(), F1Score()]
)
learn.path = Path('/kaggle/working')

# Find the best learning rate
learn.lr_find()

In [ ]:
# Fine-tune the model — freeze base, train head first, then unfreeze all- Very close to the SuggestedLRs(valley=0.0012022644514217973)
learn.fine_tune(5, base_lr=1e-3)

## Step 6 — Evaluate the Model

In [ ]:
# Show some predictions vs ground truth
learn.show_results(max_n=9, figsize=(12, 12))

In [ ]:
# Confusion matrix
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(6, 6))

# Where does the model fail most?
print('\n--- Confusion Matrix ---')
print('Rows = actual, Columns = predicted')
print('Classes:', learn.dls.vocab)

In [ ]:
# Top losses — images the model was MOST wrong about
interp.plot_top_losses(9, figsize=(15, 10))

## Step 7 — Run on the Test Set

In [ ]:
# Get predictions on the held-out test folder
test_dl = dls.test_dl(get_image_files(path/'test'))
preds, _ = learn.get_preds(dl=test_dl)

# Each prediction is [P(benign), P(malignant)]
print('Sample predictions (probability of malignant):')
for i, p in enumerate(preds[:5]):
    label = 'MALIGNANT ⚠️' if p[1] > 0.5 else 'benign ✅'
    print(f'  Image {i+1}: {p[1]:.1%} malignant → {label}')

**Error rate** is the simplest metric — it's just the percentage of predictions the model got 
wrong. Easy to understand but misleading for imbalanced datasets: if 90% of images are benign, 
a model that always predicts benign gets a 10% error rate without learning anything useful.

**ROC AUC** measures how well the model separates the two classes across all possible thresholds. 
A score of 1.0 is perfect, 0.5 is random guessing. It's threshold-independent, meaning it 
evaluates the model's overall discriminative power regardless of where you draw the line between 
benign and malignant. This is particularly valuable in medicine. The default threshold of 0.5 — 
where a probability above 0.5 is flagged as malignant — is not set in stone and should be 
discussed with a clinical team. Visualising the ROC curve helps understand the trade-off between 
**sensitivity** and **specificity** at every possible threshold, and allows clinicians to choose 
the cutoff that best fits the clinical context.

For a cancer detection model, **recall** — called **sensitivity** in medicine — matters most. 
Missing a real cancer (false negative) is far more dangerous than a false alarm (false positive). 
A model could have a decent **error rate** but terrible **recall**, which would be dangerous in 
practice. That's why **error rate** alone is not sufficient here. That said, false alarms also 
carry real consequences — suspecting cancer often means sending someone for invasive procedures 
like a biopsy. So **specificity** matters too, even if it is the secondary concern: of all the 
truly benign cases, how many did the model correctly identify as benign? The higher the 
**specificity**, the fewer healthy people are sent for unnecessary treatment.

**F1 Score** balances **precision** and **recall**. **Precision** asks: of all the cases the 
model flagged as malignant, how many actually were? **Recall** asks: of all the actual malignant 
cases, how many did the model catch? **F1** combines both into a single number between 0 and 1.

## Step 8 — Predict on a Single New Image

This is how you'd use it in a real app.

In [ ]:
def predict_mole(img_path, threshold=0.5):
    """Predict whether a mole image is benign or malignant."""
    img = PILImage.create(img_path)
    pred_class, pred_idx, probs = learn.predict(img)
    
    malignant_prob = probs[dls.vocab.o2i['malignant']].item()
    
    print(f'Image: {img_path}')
    print(f'Prediction: {pred_class}')
    print(f'Malignant risk: {malignant_prob:.1%}')
    
    if malignant_prob > 0.7:
        print('🔴 HIGH RISK — please see a dermatologist')
    elif malignant_prob > 0.4:
        print('🟡 MODERATE RISK — consider getting checked')
    else:
        print('🟢 LOW RISK — looks benign')
    
    img.show()
    return malignant_prob


# Test on a random image from the test set
test_images = get_image_files(path/'test')
sample_img = test_images[0]  # change index to try different images
predict_mole(sample_img)

The **risk** thresholds used in this **model** — low, moderate, and high — can be revisited in the same way as the probability threshold (See above). To adopt a more conservative approach, the boundary for low risk could be lowered from 0.4 to 0.2, meaning fewer cases are dismissed as benign and more are flagged for clinical review.

## Step 9 — Export the Model

Save the trained learner so you can load it later (e.g. in a Gradio or Streamlit app — as shown in your course roadmap!).

In [ ]:
learn.export('/kaggle/working/mole_classifier2.pkl')
print('Model exported to mole_classifier2.pkl ✅')
print('File size:', Path('/kaggle/working/mole_classifier2.pkl').stat().st_size // 1024, 'KB')

## General comments 
The main benchmark is the ISIC (International Skin Imaging Collaboration) challenge, which is the gold standard for this task. Here's where things stand:

**Top model performance (ISIC dataset):**


* Best competition models reach around AUC 0.93–0.95
* Winning solutions typically use EfficientNet or Vision Transformers with heavy ensembling and test-time augmentation

**vs. dermatologists:**


* A landmark 2017 Nature paper (Esteva et al.) showed that a CNN matched dermatologist performance, achieving around AUC 0.91 vs 0.86 for general dermatologists
* Specialist dermatologists with dermoscopy tools perform closer to AUC 0.89–0.91
* So top AI models now slightly exceed average dermatologist performance on controlled datasets

**This model in comparison:**

A basic ResNet34 fine-tuned for a few epochs on this smaller dataset landed around AUC 0.948, which is already better than any other model. **How?**

| Metric | Start (epoch 0) | End (epoch 4) |
|--------|----------------|---------------|
| AUC | 0.902 | 0.947 |
| Error rate | 17.3% | 13.5% |
| F1 Score | 0.817 | 0.855 |
| Valid loss | 0.465 | 0.311 |

A ResNet34 with 5 epochs on a clean binary dataset is already performing at the level of a specialist dermatologist — not because the model is exceptional, but because the problem as framed here is cleaner than real clinical conditions. 

he dataset you're using (`fanconic/skin-cancer-malignant-vs-benign`) is a relatively clean, well-curated binary dataset — much simpler than the full ISIC challenge which has 9 classes, noisier images, and much more variability. A clean binary problem is inherently easier to separate. ** **This model is NOT better than state-of-the-art models.**

### Bottom line
Important caveat: benchmark performance on clean, curated datasets doesn't always translate to real clinical settings, where image quality, lighting, and patient diversity vary enormously. That's still an open research problem.

## Model deploiment

Try this out!
[https://huggingface.co/spaces/bayaramine/mole-risk-classifier](https://huggingface.co/spaces/bayaramine/mole-risk-classifier)
